In [1]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az
from pymc.math import dot
import pytensor.tensor as pt

rng = np.random.default_rng(6420)

# 11. Brozek index prediction*

This example goes over linear regression and Bayesian $R^2$.

Adapted from [Unit 7: fat1.odc](https://raw.githubusercontent.com/areding/6420-pymc/main/original_examples/Codes4Unit7/fat2d.odc), [fat2d.odc](https://raw.githubusercontent.com/areding/6420-pymc/main/original_examples/Codes4Unit7/fat1.odc), and [fatmulti.odc](https://raw.githubusercontent.com/areding/6420-pymc/main/original_examples/Codes4Unit7/fatmulti.odc).

Data can be found [here](https://raw.githubusercontent.com/areding/6420-pymc/main/data/fat.tsv).

Percentage of body fat, age, weight, height, and ten body circumference measurements (e.g., abdomen) were recorded for 252 men. Percentage of body fat is estimated through an underwater weighing technique.

The data set has 252 observations and 15 variables. Brozek index (Brozek et al., 1963) is obtained by the underwater weighing while other 14 anthropometric variables are obtained using scales and a measuring tape.

- y = Brozek index
- X1 = 1 (intercept)
- X2 = age
- X3 = weight
- X4 = height
- X5 = adipose
- X6 = neck  
- X7 = chest
- X8 = abdomen
- X9 = hip
- X10 = thigh
- X11 = knee   
- X12 = ankle
- X13 = bicep
- X14 = forearm
- X15 = wrist

These anthropometric variables are less intrusive but also less reliable in assessing the body fat index.

Set a linear regression to predict the Brozek index from these body measurements.

## Single predictor (X8)

This is a recreation of fat1.odc.

In [2]:
data = pd.read_csv("../data/fat.tsv", sep="\t")

y = data["y"].to_numpy(copy=True)
X = data["X8"].to_numpy(copy=True)

# p will be the number of predictors + intercept (1 + 1 in this case)
n, p = X.shape[0], 2

In [3]:
with pm.Model() as m:
    tau = pm.Gamma("tau", 0.001, 0.001)
    beta0 = pm.Normal("beta0_intercept", 0, tau=0.001)
    beta1 = pm.Normal("beta1_abdomen", 0, tau=0.001)
    variance = pm.Deterministic("variance", 1 / tau)

    mu = pm.Deterministic("mu", beta0 + beta1 * X)
    likelihood = pm.Normal("likelihood", mu=mu, tau=tau, observed=y)

    # Bayesian R2 from fat1.odc
    # no need to use this anymore, use az.bayesian_r2 instead
    sse = (n - p) * variance
    cy = y - y.mean()
    sst = dot(cy, cy)
    br2 = pm.Deterministic("br2", 1 - sse / sst)

    trace = pm.sample(2000, random_seed=rng)

NUTS[nutpie]: [tau, beta0_intercept, beta1_abdomen]


Output()

In [4]:
az.summary(trace, var_names="~mu", ci_prob=0.95)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
beta0_intercept,-35,2.49,-40,-30,1266,1258,1.00,0.07,0.049
beta1_abdomen,0.583,0.027,0.53,0.64,1268,1303,1.00,0.00075,0.00052
tau,0.049,0.0044,0.041,0.058,3995,3516,1.00,7e-05,5.1e-05
variance,20.57,1.88,17,25,3995,3516,1.00,0.03,0.024
br2,0.659,0.0312,0.59,0.71,3995,3516,1.00,0.0005,0.00039


This matches the results from the U7 L11 video.

Another way to calculate the $R^2$ using a posterior predictive check (keeping in mind that there is no standard "Bayesian $R^2$"). The results will be slightly different:

In [5]:
az.bayesian_r2(
    trace,
    pred_mean="mu",
    scale="variance",
    scale_kind="var",
    ci_prob=0.95,
    ci_kind="hdi",
)

bayesian_R2(mean=0.66, hdi_lb=0.6, hdi_ub=0.71)

## Multivariate regression with all predictors

Based on fat2d.odc or fatmulti.odc (they appear to be identical). This is using Zellner's G-prior {cite:t}`zellner1986assessing`. Note that this is an uncommon prior and may not always be the best choice.

In [6]:
data = pd.read_csv("../data/fat.tsv", sep="\t")

y = data["y"].to_numpy(copy=True)
X = data.iloc[:, 1:].to_numpy(copy=True)

# add intercept
X_aug = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)
n, p = X_aug.shape

# Zellner's g
g = p**2

n, p, g

(252, 15, 225)

In [7]:
X_aug.shape

(252, 15)

In [8]:
mu_beta = np.zeros(p)

In [9]:
with pm.Model() as m2d:
    tau = pm.Gamma("tau", 0.01, 0.01)
    variance = pm.Deterministic("variance", 1 / tau)

    # tau_matrix = pt.fill(pt.zeros((15, 15)), tau)
    # tau_beta = tau_matrix / g * dot(X_aug.T, X_aug)
    tau_beta = (tau / g) * (X_aug.T @ X_aug)
    beta = pm.MvNormal("beta", mu_beta, tau=tau_beta)

    mu = pm.Deterministic("mu", dot(X_aug, beta))
    pm.Normal("likelihood", mu=mu, tau=tau, observed=y)

    # Bayesian R2 from fat2d.odc
    # No need to use this anymore! use az.bayesian_r2
    sse = (n - p) * variance
    cy = y - y.mean()
    sst = dot(cy, cy)
    br2 = pm.Deterministic("br2", 1 - sse / sst)
    br2_adj = pm.Deterministic("br2_adj", 1 - (n - 1) * variance / sst)

    trace = pm.sample(1000, random_seed=rng)

NUTS[nutpie]: [tau, beta]


Output()

In [12]:
az.summary(trace, var_names="~mu", ci_prob=0.95)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
beta[0],-15.9,16.4,-48,17,1380,1787,1.00,0.44,0.31
beta[1],0.055,0.0315,-0.0054,0.12,3595,3239,1.00,0.00053,0.00037
beta[2],-0.083,0.051,-0.18,0.02,1458,1925,1.00,0.0013,0.00093
beta[3],-0.048,0.106,-0.25,0.16,2573,2834,1.00,0.0021,0.0015
beta[4],0.07,0.29,-0.51,0.64,2670,2543,1.00,0.0056,0.0042
beta[5],-0.44,0.227,-0.89,0.0027,3014,2843,1.00,0.0041,0.003
beta[6],-0.03,0.102,-0.23,0.17,2454,2635,1.00,0.0021,0.0015
beta[7],0.874,0.089,0.7,1,2915,2270,1.00,0.0017,0.0011
beta[8],-0.197,0.139,-0.47,0.074,1731,1980,1.00,0.0033,0.0024
beta[9],0.218,0.142,-0.055,0.5,2883,2904,1.00,0.0026,0.0019


In [13]:
az.bayesian_r2(
    trace,
    pred_mean="mu",
    scale="variance",
    scale_kind="var",
    ci_prob=0.95,
    ci_kind="hdi",
)

bayesian_R2(mean=0.73, hdi_lb=0.68, hdi_ub=0.77)

In [14]:
%load_ext watermark
%watermark -n -u -v -iv

Last updated: Sun, 18 Oct 2026

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.16.1

arviz   : 1.3.0
numpy   : 2.4.6
pandas  : 3.0.5
pymc    : 6.3.1
pytensor: 3.3.0

